In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

# Search for model config files
print("🔍 Searching for model config.json files...")
print("-" * 50)

for root, dirs, files in os.walk('/kaggle/input/'):
    if 'config.json' in files:
        print(f"✓ Found model at: {root}")
        print(f"  Files: {[f for f in files if f.endswith(('.json', '.safetensors', '.bin'))]}")
        print()

In [ ]:
# =============================================================================
# Milestone 5 - Question 1: DeBERTa Highest Probability for Row 25
# =============================================================================

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# LOAD MODELS FROM HUGGINGFACE
# -------------------------------------------------------------------------
DEBERTA_PATH = 'microsoft/deberta-v3-small'

print("Loading DeBERTa...")
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, num_labels=5)
deberta_model.to(device)
deberta_model.eval()

# -------------------------------------------------------------------------
# ⚠️ PASTE YOUR CORRECT PATH HERE ⚠️
# -------------------------------------------------------------------------
# Example: '/kaggle/input/my-dataset/train.csv'
# Example: '/kaggle/working/train (4).csv'
TRAIN_CSV_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv' 

# -------------------------------------------------------------------------
# Load Data
# -------------------------------------------------------------------------
train_df = pd.read_csv(TRAIN_CSV_PATH)

# Get Row 25
row = train_df.iloc[25]
prompt = row['prompt']
options = {
    'A': row['A'],
    'B': row['B'],
    'C': row['C'],
    'D': row['D'],
    'E': row['E']
}

# -------------------------------------------------------------------------
# Label Mapping
# -------------------------------------------------------------------------
label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# -------------------------------------------------------------------------
# Format Input
# -------------------------------------------------------------------------
def format_mcq_input(prompt, options_dict, tokenizer):
    sep = tokenizer.sep_token if tokenizer.sep_token else "[SEP]"
    formatted = f"{prompt}"
    for opt_label in ['A', 'B', 'C', 'D', 'E']:
        formatted += f" {sep} {opt_label}. {options_dict[opt_label]}"
    return formatted

# -------------------------------------------------------------------------
# DeBERTa Inference on Row 25
# -------------------------------------------------------------------------
input_text = format_mcq_input(prompt, options, deberta_tokenizer)

inputs = deberta_tokenizer(
    input_text,
    return_tensors='pt',
    truncation=True,
    max_length=1024,
    padding=True
).to(device)

with torch.no_grad():
    outputs = deberta_model(**inputs)
    logits = outputs.logits

# Apply Softmax
probs = F.softmax(logits, dim=-1)
probs_np = probs.cpu().numpy()[0]

# Display results
print("\n" + "=" * 50)
print("DeBERTa PROBABILITIES FOR ROW 25")
print("=" * 50)

for i, (label, prob) in enumerate(zip(['A', 'B', 'C', 'D', 'E'], probs_np)):
    marker = " ◀ HIGHEST" if i == np.argmax(probs_np) else ""
    print(f"  {label}: {prob:.4f}{marker}")

# Get Answer for Question 1
max_idx = np.argmax(probs_np)
max_prob = probs_np[max_idx]
highest_option = label_map[max_idx]

print("-" * 50)
print(f"✅ ANSWER: {highest_option}, {max_prob:.4f}")
print("=" * 50)

In [ ]:
# =============================================================================
# Milestone 5 - Question 2: Simple Ensemble (Average Probabilities) for Row 25
# =============================================================================

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import glob
import os

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# Step 1: Auto-find the train.csv path
# -------------------------------------------------------------------------
print("🔍 Searching for train.csv...")
matches = glob.glob('/kaggle/**/train*.csv', recursive=True)
if matches:
    TRAIN_CSV_PATH = matches[0]
    print(f"✅ Found: {TRAIN_CSV_PATH}\n")
else:
    raise FileNotFoundError("Could not find train.csv. Please add the dataset to your notebook.")

# -------------------------------------------------------------------------
# Step 2: Load Models from HuggingFace
# -------------------------------------------------------------------------
DEBERTA_PATH = 'microsoft/deberta-v3-small'
ROBERTA_PATH = 'roberta-base'

print("Loading DeBERTa...")
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, num_labels=5)
deberta_model.to(device)
deberta_model.eval()

print("Loading RoBERTa...")
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH, num_labels=5)
roberta_model.to(device)
roberta_model.eval()

# -------------------------------------------------------------------------
# Step 3: Load Data and get Row 25
# -------------------------------------------------------------------------
train_df = pd.read_csv(TRAIN_CSV_PATH)
row = train_df.iloc[25]

prompt = row['prompt']
options = {
    'A': row['A'], 'B': row['B'], 'C': row['C'], 
    'D': row['D'], 'E': row['E']
}

label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# -------------------------------------------------------------------------
# Step 4: Format Input
# -------------------------------------------------------------------------
def format_mcq_input(prompt, options_dict):
    # Standard format that works well for both
    formatted = f"{prompt} [SEP] "
    formatted += " [SEP] ".join([f"{k}. {v}" for k, v in options_dict.items()])
    return formatted

input_text = format_mcq_input(prompt, options)

# -------------------------------------------------------------------------
# Step 5: Inference Helper Function
# -------------------------------------------------------------------------
def get_probabilities(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=1024, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
    return probs

deberta_probs = get_probabilities(deberta_model, deberta_tokenizer, input_text)
roberta_probs = get_probabilities(roberta_model, roberta_tokenizer, input_text)

# -------------------------------------------------------------------------
# Step 6: Simple Ensemble (Average Probabilities)
# -------------------------------------------------------------------------
avg_probs = (deberta_probs + roberta_probs) / 2.0

# -------------------------------------------------------------------------
# Step 7: Display Results & Answer
# -------------------------------------------------------------------------
print("=" * 70)
print("ENSEMBLE PROBABILITIES FOR ROW 25")
print("=" * 70)
print(f"{'Option':<8} | {'DeBERTa':<10} | {'RoBERTa':<10} | {'Average':<10}")
print("-" * 70)

max_avg_prob = 0
best_option = ""

for i, label in enumerate(['A', 'B', 'C', 'D', 'E']):
    d_prob = deberta_probs[i]
    r_prob = roberta_probs[i]
    a_prob = avg_probs[i]
    
    marker = " ◀ HIGHEST" if a_prob == max(avg_probs) else ""
    print(f"{label:<8} | {d_prob:<10.4f} | {r_prob:<10.4f} | {a_prob:<10.4f}{marker}")

print("-" * 70)

# Final Answer extraction
max_idx = np.argmax(avg_probs)
highest_option = label_map[max_idx]
max_prob = avg_probs[max_idx]

print(f"\n✅ QUESTION 2 ANSWER: {highest_option}")
print("=" * 70)

In [ ]:
# =============================================================================
# Milestone 5 - Question 3: Weighted Ensemble (0.7 DeBERTa + 0.3 RoBERTa)
# =============================================================================

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import glob

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# Step 1: Auto-find the train.csv path
# -------------------------------------------------------------------------
print("🔍 Searching for train.csv...")
matches = glob.glob('/kaggle/**/train*.csv', recursive=True)
if matches:
    TRAIN_CSV_PATH = matches[0]
    print(f"✅ Found: {TRAIN_CSV_PATH}\n")
else:
    raise FileNotFoundError("Could not find train.csv. Please add the dataset to your notebook.")

# -------------------------------------------------------------------------
# Step 2: Load Models from HuggingFace
# -------------------------------------------------------------------------
DEBERTA_PATH = 'microsoft/deberta-v3-small'
ROBERTA_PATH = 'roberta-base'

print("Loading DeBERTa...")
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, num_labels=5)
deberta_model.to(device)
deberta_model.eval()

print("Loading RoBERTa...")
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH, num_labels=5)
roberta_model.to(device)
roberta_model.eval()

# -------------------------------------------------------------------------
# Step 3: Load Data and get Row 25
# -------------------------------------------------------------------------
train_df = pd.read_csv(TRAIN_CSV_PATH)
row = train_df.iloc[25]

prompt = row['prompt']
options = {
    'A': row['A'], 'B': row['B'], 'C': row['C'], 
    'D': row['D'], 'E': row['E']
}

label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# -------------------------------------------------------------------------
# Step 4: Format Input
# -------------------------------------------------------------------------
def format_mcq_input(prompt, options_dict):
    formatted = f"{prompt} [SEP] "
    formatted += " [SEP] ".join([f"{k}. {v}" for k, v in options_dict.items()])
    return formatted

input_text = format_mcq_input(prompt, options)

# -------------------------------------------------------------------------
# Step 5: Inference Helper Function
# -------------------------------------------------------------------------
def get_probabilities(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=1024, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
    return probs

deberta_probs = get_probabilities(deberta_model, deberta_tokenizer, input_text)
roberta_probs = get_probabilities(roberta_model, roberta_tokenizer, input_text)

# -------------------------------------------------------------------------
# Step 6: Weighted Ensemble
# -------------------------------------------------------------------------
W_DEBERTA = 0.70
W_ROBERTA = 0.30

# P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]
weighted_probs = (W_DEBERTA * deberta_probs) + (W_ROBERTA * roberta_probs)

# -------------------------------------------------------------------------
# Step 7: Display Results & Answer
# -------------------------------------------------------------------------
print("=" * 75)
print("WEIGHTED ENSEMBLE PROBABILITIES FOR ROW 25")
print("=" * 75)
print(f"{'Option':<8} | {'DeBERTa':<10} | {'RoBERTa':<10} | {'Weighted (0.7/0.3)':<18}")
print("-" * 75)

for i, label in enumerate(['A', 'B', 'C', 'D', 'E']):
    d_prob = deberta_probs[i]
    r_prob = roberta_probs[i]
    w_prob = weighted_probs[i]
    
    marker = " ◀ RANK 1" if w_prob == max(weighted_probs) else ""
    print(f"{label:<8} | {d_prob:<10.4f} | {r_prob:<10.4f} | {w_prob:<18.4f}{marker}")

print("-" * 75)

# Final Answer extraction
max_idx = np.argmax(weighted_probs)
highest_option = label_map[max_idx]

print(f"\n✅ QUESTION 3 ANSWER: {highest_option}")
print("=" * 75)

In [ ]:
# =============================================================================
# Milestone 5 - Question 4: Top-3 Prediction String for Row 25
# =============================================================================

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import glob

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# Step 1: Auto-find the train.csv path
# -------------------------------------------------------------------------
print("🔍 Searching for train.csv...")
matches = glob.glob('/kaggle/**/train*.csv', recursive=True)
if matches:
    TRAIN_CSV_PATH = matches[0]
    print(f"✅ Found: {TRAIN_CSV_PATH}\n")
else:
    raise FileNotFoundError("Could not find train.csv.")

# -------------------------------------------------------------------------
# Step 2: Load Models
# -------------------------------------------------------------------------
DEBERTA_PATH = 'microsoft/deberta-v3-small'
ROBERTA_PATH = 'roberta-base'

print("Loading models...")
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, num_labels=5).to(device).eval()

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH, num_labels=5).to(device).eval()

# -------------------------------------------------------------------------
# Step 3: Load Data and get Row 25
# -------------------------------------------------------------------------
train_df = pd.read_csv(TRAIN_CSV_PATH)
row = train_df.iloc[25]
prompt = row['prompt']
options = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D'], 'E': row['E']}
label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# -------------------------------------------------------------------------
# Step 4: Inference & Weighted Ensembling
# -------------------------------------------------------------------------
def format_mcq_input(prompt, options_dict):
    formatted = f"{prompt} [SEP] "
    formatted += " [SEP] ".join([f"{k}. {v}" for k, v in options_dict.items()])
    return formatted

def get_probabilities(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=1024, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return F.softmax(logits, dim=-1).cpu().numpy()[0]

input_text = format_mcq_input(prompt, options)
deberta_probs = get_probabilities(deberta_model, deberta_tokenizer, input_text)
roberta_probs = get_probabilities(roberta_model, roberta_tokenizer, input_text)

# Apply Weights (0.7 DeBERTa + 0.3 RoBERTa)
weighted_probs = (0.70 * deberta_probs) + (0.30 * roberta_probs)

# -------------------------------------------------------------------------
# Step 5: Rank Options and Extract Top-3 String
# -------------------------------------------------------------------------
# Get indices that would sort the array from highest to lowest probability
ranked_indices = np.argsort(weighted_probs)[::-1]

# Map indices to labels
ranked_labels = [label_map[idx] for idx in ranked_indices]

# Extract the top 3
top_3_labels = ranked_labels[:3]

# Join with space for Kaggle submission format
prediction_string = " ".join(top_3_labels)

# -------------------------------------------------------------------------
# Step 6: Display Results
# -------------------------------------------------------------------------
print("=" * 60)
print("FULL RANKING FOR ROW 25 (Weighted Ensemble)")
print("=" * 60)
print(f"{'Rank':<6} | {'Option':<8} | {'Weighted Prob'}")
print("-" * 60)

for rank, idx in enumerate(ranked_indices, 1):
    label = label_map[idx]
    prob = weighted_probs[idx]
    in_top_3 = " <-- TOP 3" if rank <= 3 else ""
    print(f"{rank:<6} | {label:<8} | {prob:.4f}{in_top_3}")

print("-" * 60)
print(f"\n✅ QUESTION 4 ANSWER: {prediction_string}")
print("=" * 60)

In [ ]:
# =============================================================================
# Milestone 5 - Question 5 (FIXED): Generate submission.csv
# =============================================================================

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import glob
from tqdm import tqdm
import os

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# -------------------------------------------------------------------------
# Step 1: Auto-find CSV path
# -------------------------------------------------------------------------
print("🔍 Searching for test.csv...")
test_matches = glob.glob('/kaggle/**/test*.csv', recursive=True)
if test_matches:
    TEST_CSV_PATH = test_matches[0]
    print(f"✅ Found: {TEST_CSV_PATH}\n")
else:
    raise FileNotFoundError("Could not find test.csv.")

# -------------------------------------------------------------------------
# Step 2: Load Models (FIXED MAX LENGTH = 512)
# -------------------------------------------------------------------------
DEBERTA_PATH = 'microsoft/deberta-v3-small'
ROBERTA_PATH = 'roberta-base'

# Safe max length for DeBERTa v3
MAX_LEN = 512 

print("Loading DeBERTa...")
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, num_labels=5).to(device).eval()

print("Loading RoBERTa...")
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTa_PATH)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH, num_labels=5).to(device).eval()

# -------------------------------------------------------------------------
# Step 3: Load Test Data
# -------------------------------------------------------------------------
test_df = pd.read_csv(TEST_CSV_PATH)
label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# -------------------------------------------------------------------------
# Step 4: Helper Functions
# -------------------------------------------------------------------------
def format_mcq_input(prompt, options_dict):
    formatted = f"{prompt} [SEP] "
    formatted += " [SEP] ".join([f"{k}. {v}" for k, v in options_dict.items()])
    return formatted

def get_probabilities(model, tokenizer, text):
    # KEY FIX: Truncate to MAX_LEN to prevent CUDA Assert
    inputs = tokenizer(
        text, 
        return_tensors='pt', 
        truncation=True, 
        max_length=MAX_LEN, 
        padding='max_length' # Ensures uniform tensor shapes
    ).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    return F.softmax(logits, dim=-1).cpu().numpy()[0]

# -------------------------------------------------------------------------
# Step 5: Run Pipeline on Entire Test Set
# -------------------------------------------------------------------------
predictions = []

print(f"\n🚀 Running weighted ensemble (0.7 DeBERTa / 0.3 RoBERTa) on {len(test_df)} rows...")
print("-" * 60)

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        prompt = row['prompt']
        options = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D'], 'E': row['E']}
        
        input_text = format_mcq_input(prompt, options)
        
        deberta_probs = get_probabilities(deberta_model, deberta_tokenizer, input_text)
        roberta_probs = get_probabilities(roberta_model, roberta_tokenizer, input_text)
        
        # Weighted Ensemble
        weighted_probs = (0.70 * deberta_probs) + (0.30 * roberta_probs)
        
        # Get Top-3
        top_3_idx = np.argsort(weighted_probs)[::-1][:3]
        top_3_labels = [label_map[i] for i in top_3_idx]
        pred_string = " ".join(top_3_labels)
        
        predictions.append({'id': row['id'], 'prediction': pred_string})
        
    except Exception as e:
        print(f"\nError on row {row['id']}: {e}")
        # Fallback to random prediction if a specific row still fails
        predictions.append({'id': row['id'], 'prediction': 'A B C'})

# -------------------------------------------------------------------------
# Step 6: Save and Count
# -------------------------------------------------------------------------
sub_df = pd.DataFrame(predictions)
sub_df.to_csv('submission.csv', index=False)

row_count = len(sub_df)

print("\n" + "=" * 60)
print("submission.csv PREVIEW:")
print("=" * 60)
print(sub_df.head())
print("...")
print(sub_df.tail())
print("=" * 60)
print(f"✅ QUESTION 5 ANSWER: {row_count}")
print("=" * 60)

In [ ]:
# =============================================================================
# Milestone 5 - Question 6: Test-Time Augmentation (TTA) Top-1 Changes
# =============================================================================

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import glob

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# Step 1: Load Model (DeBERTa only, with MAX_LEN=512 to prevent crashes)
# -------------------------------------------------------------------------
DEBERTA_PATH = 'microsoft/deberta-v3-small'
MAX_LEN = 512

print("Loading DeBERTa...")
tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, num_labels=5).to(device).eval()

# -------------------------------------------------------------------------
# Step 2: Load First 50 Rows of Test Data
# -------------------------------------------------------------------------
print("🔍 Searching for test.csv...")
test_matches = glob.glob('/kaggle/**/test*.csv', recursive=True)
if test_matches:
    TEST_CSV_PATH = test_matches[0]
else:
    raise FileNotFoundError("Could not find test.csv.")

test_df = pd.read_csv(TEST_CSV_PATH).head(50) # ONLY FIRST 50 ROWS
label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# TTA Prefix
TTA_PREFIX = "Answer the following multiple-choice question carefully: "

# -------------------------------------------------------------------------
# Step 3: Helper Functions
# -------------------------------------------------------------------------
def format_mcq(prompt, options_dict):
    formatted = f"{prompt} [SEP] "
    formatted += " [SEP] ".join([f"{k}. {v}" for k, v in options_dict.items()])
    return formatted

def get_probabilities(text):
    inputs = tokenizer(
        text, 
        return_tensors='pt', 
        truncation=True, 
        max_length=MAX_LEN, 
        padding='max_length'
    ).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    return F.softmax(logits, dim=-1).cpu().numpy()[0]

# -------------------------------------------------------------------------
# Step 4: Run TTA Comparison
# -------------------------------------------------------------------------
different_top1_count = 0

print(f"\n🚀 Running TTA on first 50 rows...")
print("-" * 60)

for index, row in test_df.iterrows():
    options = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D'], 'E': row['E']}
    
    # 1. Original Prompt Inference
    text_orig = format_mcq(row['prompt'], options)
    probs_orig = get_probabilities(text_orig)
    top1_orig = np.argmax(probs_orig)
    
    # 2. Augmented Prompt Inference
    text_aug = format_mcq(TTA_PREFIX + row['prompt'], options)
    probs_aug = get_probabilities(text_aug)
    
    # 3. Average Probabilities (TTA)
    probs_tta = (probs_orig + probs_aug) / 2.0
    top1_tta = np.argmax(probs_tta)
    
    # 4. Check if Top-1 changed
    if top1_orig != top1_tta:
        different_top1_count += 1
        print(f"Row {row['id']}: Changed from {label_map[top1_orig]} -> {label_map[top1_tta]}")

print("-" * 60)

# -------------------------------------------------------------------------
# Step 5: Output Answer
# -------------------------------------------------------------------------
print(f"\n✅ QUESTION 6 ANSWER: {different_top1_count}")
print("=" * 60)

In [ ]:
# =============================================================================
# Milestone 5 - Question 7: DeBERTa vs Weighted Ensemble Top-1 Differences
# =============================================================================

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import glob
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# Step 1: Load Models (with MAX_LEN=512 fix)
# -------------------------------------------------------------------------
DEBERTA_PATH = 'microsoft/deberta-v3-small'
ROBERTA_PATH = 'roberta-base'
MAX_LEN = 512

print("Loading DeBERTa...")
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, num_labels=5).to(device).eval()

print("Loading RoBERTa...")
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH, num_labels=5).to(device).eval()

# -------------------------------------------------------------------------
# Step 2: Load First 100 Rows of Test Data
# -------------------------------------------------------------------------
print("🔍 Searching for test.csv...")
test_matches = glob.glob('/kaggle/**/test*.csv', recursive=True)
if test_matches:
    TEST_CSV_PATH = test_matches[0]
else:
    raise FileNotFoundError("Could not find test.csv.")

test_df = pd.read_csv(TEST_CSV_PATH).head(100) # ONLY FIRST 100 ROWS
label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# -------------------------------------------------------------------------
# Step 3: Helper Functions
# -------------------------------------------------------------------------
def format_mcq(prompt, options_dict):
    formatted = f"{prompt} [SEP] "
    formatted += " [SEP] ".join([f"{k}. {v}" for k, v in options_dict.items()])
    return formatted

def get_probabilities(model, tokenizer, text):
    inputs = tokenizer(
        text, 
        return_tensors='pt', 
        truncation=True, 
        max_length=MAX_LEN, 
        padding='max_length'
    ).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    return F.softmax(logits, dim=-1).cpu().numpy()[0]

# -------------------------------------------------------------------------
# Step 4: Run Comparison
# -------------------------------------------------------------------------
different_predictions_count = 0

print(f"\n🚀 Comparing DeBERTa vs Weighted Ensemble on first 100 rows...")
print("-" * 60)

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    options = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D'], 'E': row['E']}
    input_text = format_mcq(row['prompt'], options)
    
    # 1. Get DeBERTa Top-1
    deberta_probs = get_probabilities(deberta_model, deberta_tokenizer, input_text)
    top1_deberta = np.argmax(deberta_probs)
    
    # 2. Get Weighted Ensemble Top-1
    roberta_probs = get_probabilities(roberta_model, roberta_tokenizer, input_text)
    weighted_probs = (0.70 * deberta_probs) + (0.30 * roberta_probs)
    top1_ensemble = np.argmax(weighted_probs)
    
    # 3. Compare
    if top1_deberta != top1_ensemble:
        different_predictions_count += 1
        print(f"Row {row['id']}: DeBERTa={label_map[top1_deberta]} | Ensemble={label_map[top1_ensemble]}")

print("-" * 60)

# -------------------------------------------------------------------------
# Step 5: Output Answer
# -------------------------------------------------------------------------
print(f"\n✅ QUESTION 7 ANSWER: {different_predictions_count}")
print("=" * 60)

In [ ]:
# =============================================================================
# Milestone 5 - Question 8: Positive Confidence Gain Count
# =============================================================================

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import glob
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# Step 1: Load Models (with MAX_LEN=512 fix)
# -------------------------------------------------------------------------
DEBERTA_PATH = 'microsoft/deberta-v3-small'
ROBERTA_PATH = 'roberta-base'
MAX_LEN = 512

print("Loading DeBERTa...")
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, num_labels=5).to(device).eval()

print("Loading RoBERTa...")
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH, num_labels=5).to(device).eval()

# -------------------------------------------------------------------------
# Step 2

In [ ]:
# =============================================================================
# Milestone 5 - Question 9: Top-3 Ranking Changes Count
# =============================================================================

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import glob
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# Step 1: Load Models (with MAX_LEN=512 fix)
# -------------------------------------------------------------------------
DEBERTA_PATH = 'microsoft/deberta-v3-small'
ROBERTA_PATH = 'roberta-base'
MAX_LEN = 512

print("Loading DeBERTa...")
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, num_labels=5).to(device).eval()

print("Loading RoBERTa...")
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH, num_labels=5).to(device).eval()

# -------------------------------------------------------------------------
# Step 2: Load First 100 Rows of Test Data
# -------------------------------------------------------------------------
print("🔍 Searching for test.csv...")
test_matches = glob.glob('/kaggle/**/test*.csv', recursive=True)
if test_matches:
    TEST_CSV_PATH = test_matches[0]
else:
    raise FileNotFoundError("Could not find test.csv.")

test_df = pd.read_csv(TEST_CSV_PATH).head(100) # ONLY FIRST 100 ROWS
label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# -------------------------------------------------------------------------
# Step 3: Helper Functions
# -------------------------------------------------------------------------
def format_mcq(prompt, options_dict):
    formatted = f"{prompt} [SEP] "
    formatted += " [SEP] ".join([f"{k}. {v}" for k, v in options_dict.items()])
    return formatted

def get_probabilities(model, tokenizer, text):
    inputs = tokenizer(
        text, 
        return_tensors='pt', 
        truncation=True, 
        max_length=MAX_LEN, 
        padding='max_length'
    ).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    return F.softmax(logits, dim=-1).cpu().numpy()[0]

def get_top3_string(probs):
    # Get indices of top 3 highest probabilities
    top_3_idx = np.argsort(probs)[::-1][:3]
    # Map to labels and join with space
    return " ".join([label_map[i] for i in top_3_idx])

# -------------------------------------------------------------------------
# Step 4: Run Comparison
# -------------------------------------------------------------------------
ranking_changes_count = 0

print(f"\n🚀 Comparing Top-3 Rankings (DeBERTa vs Ensemble) on first 100 rows...")
print("-" * 70)
print(f"{'Row ID':<8} | {'DeBERTa Top-3':<16} | {'Ensemble Top-3':<16}")
print("-" * 70)

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    options = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D'], 'E': row['E']}
    input_text = format_mcq(row['prompt'], options)
    
    # 1. DeBERTa Top-3 String
    deberta_probs = get_probabilities(deberta_model, deberta_tokenizer, input_text)
    deberta_top3 = get_top3_string(deberta_probs)
    
    # 2. Weighted Ensemble Top-3 String
    roberta_probs = get_probabilities(roberta_model, roberta_tokenizer, input_text)
    weighted_probs = (0.70 * deberta_probs) + (0.30 * roberta_probs)
    ensemble_top3 = get_top3_string(weighted_probs)
    
    # 3. Compare Strings
    if deberta_top3 != ensemble_top3:
        ranking_changes_count += 1
        print(f"{row['id']:<8} | {deberta_top3:<16} | {ensemble_top3:<16}")

print("-" * 70)

# -------------------------------------------------------------------------
# Step 5: Output Answer
# -------------------------------------------------------------------------
print(f"\n✅ QUESTION 9 ANSWER: {ranking_changes_count}")
print("=" * 70)

In [ ]:
# =============================================================================
# Milestone 5 - Question 10: Compute Final MAP@3 Score
# =============================================================================

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import glob
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# Step 1: Load Models (with MAX_LEN=512 fix)
# -------------------------------------------------------------------------
DEBERTA_PATH = 'microsoft/deberta-v3-small'
ROBERTA_PATH = 'roberta-base'
MAX_LEN = 512

print("Loading DeBERTa...")
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, num_labels=5).to(device).eval()

print("Loading RoBERTa...")
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH, num_labels=5).to(device).eval()

# -------------------------------------------------------------------------
# Step 2: Load First 100 Rows of Train Data (for Ground Truth)
# -------------------------------------------------------------------------
print("🔍 Searching for train.csv...")
train_matches = glob.glob('/kaggle/**/train*.csv', recursive=True)
if train_matches:
    TRAIN_CSV_PATH = train_matches[0]
else:
    raise FileNotFoundError("Could not find train.csv.")

# Using train.csv because we need the 'answer' column to calculate MAP@3
df = pd.read_csv(TRAIN_CSV_PATH).head(100) 
label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# -------------------------------------------------------------------------
# Step 3: Helper Functions
# -------------------------------------------------------------------------
def format_mcq(prompt, options_dict):
    formatted = f"{prompt} [SEP] "
    formatted += " [SEP] ".join([f"{k}. {v}" for k, v in options_dict.items()])
    return formatted

def get_probabilities(model, tokenizer, text):
    inputs = tokenizer(
        text, 
        return_tensors='pt', 
        truncation=True, 
        max_length=MAX_LEN, 
        padding='max_length'
    ).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    return F.softmax(logits, dim=-1).cpu().numpy()[0]

def get_top3_string(probs):
    top_3_idx = np.argsort(probs)[::-1][:3]
    return " ".join([label_map[i] for i in top_3_idx])

# -------------------------------------------------------------------------
# Step 4: MAP@3 Calculation Function
# -------------------------------------------------------------------------
def calculate_map_at_3(ground_truth, predictions):
    """
    Calculates Mean Average Precision at 3 (MAP@3)
    """
    ap_scores = []
    
    for true_answer, pred_string in zip(ground_truth, predictions):
        pred_labels = pred_string.split()
        
        try:
            # Find 1-based index of the true answer in the top 3 predictions
            rank = pred_labels.index(true_answer) + 1
            # AP@3 = 1 / rank
            ap_scores.append(1.0 / rank)
        except ValueError:
            # If true answer is not in top 3, score is 0
            ap_scores.append(0.0)
            
    # Mean of all AP scores
    return np.mean(ap_scores)

# -------------------------------------------------------------------------
# Step 5: Run Pipeline & Compute Score
# -------------------------------------------------------------------------
predictions = []
ground_truth = df['answer'].tolist()

print(f"\n🚀 Running Weighted Ensemble on first 100 validation samples...")
print("-" * 60)

for index, row in tqdm(df.iterrows(), total=len(df)):
    options = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D'], 'E': row['E']}
    input_text = format_mcq(row['prompt'], options)
    
    # Weighted Ensemble
    deberta_probs = get_probabilities(deberta_model, deberta_tokenizer, input_text)
    roberta_probs = get_probabilities(roberta_model, roberta_tokenizer, input_text)
    weighted_probs = (0.70 * deberta_probs) + (0.30 * roberta_probs)
    
    # Get Top-3 String
    pred_string = get_top3_string(weighted_probs)
    predictions.append(pred_string)

# -------------------------------------------------------------------------
# Step 6: Output Answer
# -------------------------------------------------------------------------
final_map3 = calculate_map_at_3(ground_truth, predictions)

print("-" * 60)
print("\nSample Predictions vs Ground Truth:")
for i in range(5):
    print(f"  GT: {ground_truth[i]} | Pred: {predictions[i]}")

print("\n" + "=" * 60)
print(f"✅ QUESTION 10 ANSWER: {final_map3:.4f}")
print("=" * 60)